# SageMaker Init

In [1]:
import time

from pathlib import Path
from datasets import load_from_disk
from tqdm.notebook import tqdm

import sagemaker

from config.settings import AWSSettings, DatasetSettings
from sagemaker.huggingface import HuggingFace
from sagemaker.debugger import TensorBoardOutputConfig
from sagemaker.huggingface.model import HuggingFaceModel

/home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[03/25/25 19:06:31] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=866667;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=302011;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py#1352\1352]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ssm-user/.config/sagemaker/config.yaml


In [2]:
aws_settings = AWSSettings()
dataset_settings = DatasetSettings()

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Get the AWS region
region = sagemaker_session.boto_region_name
print(f"SageMaker running in region: {region}")
print(f"SageMaker role ARN: {aws_settings.EXECUTION_ROLE}")
print(f"SageMaker bucket: {aws_settings.BUCKET}")

[03/25/25 19:06:53] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=697083;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=883233;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py#1352\1352]8;;\

SageMaker running in region: eu-central-1
SageMaker role ARN: arn:aws:iam::421646001410:role/service-role/AmazonSageMaker-ExecutionRole-20210811T103532
SageMaker bucket: pivanov-tac-bucket


## TinyBERT

In [3]:
model_package = "s3://pivanov-tac-bucket/models/tinybert/model.tar.gz"

In [4]:
tinybert = HuggingFaceModel(
    model_data=model_package,
    role=aws_settings.EXECUTION_ROLE,
    transformers_version="4.6.1",
    pytorch_version="1.7.1",
    py_version="py36",
)

[03/25/25 19:07:40] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=246427;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=575765;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py#1352\1352]8;;\

In [5]:
predictor = tinybert.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=f"tac-sentiment-analysis-tinybert-{time.strftime('%Y-%m-%d-%H-%M', time.gmtime())}"
)

[03/25/25 19:08:04] INFO     Creating model with name:                                              ]8;id=958775;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=717700;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#4094\4094]8;;\
                             huggingface-pytorch-inference-2025-03-25-18-08-04-408                                 

[03/25/25 19:08:05] INFO     Creating endpoint-config with name                                     ]8;id=933598;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=28483;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#5937\5937]8;;\
                             tac-sentiment-analysis-tinybert-2025-03-25-18-08                                      

                    INFO     Creating endpoint with name                                            ]8;id=163648;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=182404;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#4759\4759]8;;\
                             tac-sentiment-analysis-tinybert-2025-03-25-18-08                                      

-------!

In [8]:
# example request: you always need to define "inputs"
data = {
   "inputs": "Camera - You are awarded a SiPix Digital Camera! call 09061221066 fromm landline. Delivery within 28 days."
}

# request
predictor.predict(
    data,
    initial_args={"ContentType": "application/json"},
)

[{'label': 'positive', 'score': 0.9111132025718689}]

## RoBERTa

In [9]:
model_package = "s3://pivanov-tac-bucket/models/roberta-base/model.tar.gz"

In [18]:
roberta = HuggingFaceModel(
    model_data=model_package,
    role=aws_settings.EXECUTION_ROLE,
    transformers_version="4.6.1",
    pytorch_version="1.7.1",
    py_version="py36",
)

[03/25/25 20:05:12] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=24138;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=455184;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py#1352\1352]8;;\

In [19]:
predictor_roberta = roberta.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=f"tac-sentiment-analysis-roberta-base-{time.strftime('%Y-%m-%d-%H-%M', time.gmtime())}"
)

[03/25/25 20:05:18] INFO     Creating model with name:                                              ]8;id=639869;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=46287;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#4094\4094]8;;\
                             huggingface-pytorch-inference-2025-03-25-19-05-18-333                                 

[03/25/25 20:05:19] INFO     Creating endpoint-config with name                                     ]8;id=585565;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=321106;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#5937\5937]8;;\
                             tac-sentiment-analysis-roberta-base-2025-03-25-19-05                                  

                    INFO     Creating endpoint with name                                            ]8;id=523747;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=185695;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#4759\4759]8;;\
                             tac-sentiment-analysis-roberta-base-2025-03-25-19-05                                  

-------!

In [22]:
# example request: you always need to define "inputs"
data = {
   "inputs": []
}

# request
predictor_roberta.predict(
    data,
    initial_args={"ContentType": "application/json"},
)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:7                                                                                    │
│                                                                                                  │
│    4 }                                                                                           │
│    5                                                                                             │
│    6 # request                                                                                   │
│ ❱  7 predictor_roberta.predict(                                                                  │
│    8 │   data,                                                                                   │
│    9 │   initial_args={"ContentType": "application/json"},                                       │
│   10 )                                                                                           │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/base_predictor.py:212 in    │
│ predict                                                                                          │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/client.py:570 in _api_call   │
│                                                                                                  │
│    567 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    568 │   │   │   │   )                                                                         │
│    569 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  570 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    571 │   │                                                                                     │
│    572 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    573                                                                                           │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/context.py:124 in wrapper    │
│                                                                                                  │
│   121 │   │   │   with start_as_current_context():                                               │
│   122 │   │   │   │   if hook:                                                                   │
│   123 │   │   │   │   │   hook()                                                                 │
│ ❱ 124 │   │   │   │   return func(*args, **kwargs)                                               │
│   125 │   │                                                                                      │
│   126 │   │   return wrapper                               

{"ErrorCode":"CLIENT_ERROR_FROM_MODEL","LogStreamArn":"arn:aws:logs:eu-central-1:421646001410:log-group:/aws/sagemaker/Endpoints/tac-sentiment-analysis-roberta-base-2025-03-25-19-05","Message":"Received client error (400) from primary with message \"{\n  \"code\": 400,\n  \"type\": \"InternalServerException\",\n  \"message\": \"list index out of range\"\n}\n\". See https://eu-central-1.console.aws.amazon.com/cloudwatch/home?region=eu-central-1#logEventViewer:group=/aws/sagemaker/Endpoints/tac-sentiment-analysis-roberta-base-2025-03-25-19-05 in account 421646001410 for more information.","OriginalMessage":"{\n  \"code\": 400,\n  \"type\": \"InternalServerException\",\n  \"message\": \"list index out of range\"\n}\n","OriginalStatusCode":400}

In [ ]:
# # Delete the endpoint
# sagemaker_session.delete_endpoint(predictor_roberta.endpoint_name)

# # Delete the endpoint configuration
# sagemaker_session.delete_endpoint_config(predictor_roberta.endpoint_name)

# # Delete the model
# sagemaker_session.delete_model(roberta.name)

[03/25/25 20:04:15] INFO     Deleting endpoint with name:                                           ]8;id=980190;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=719983;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#4903\4903]8;;\
                             tac-sentiment-analysis-roberta-base-2025-03-25-18-29                                  

                    INFO     Deleting endpoint configuration with name:                             ]8;id=272511;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=896982;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#4913\4913]8;;\
                             tac-sentiment-analysis-roberta-base-2025-03-25-18-29                                  

[03/25/25 20:04:16] INFO     Deleting model with name:                                              ]8;id=323593;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=939805;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#5274\5274]8;;\
                             huggingface-pytorch-inference-2025-03-25-18-29-23-500                                 